### Improve script (script4a.py) to convert bulk listing file from amazon to shopify

In [1]:
import os
import io
import pandas as pd
import numpy as np
from loguru import logger

In [2]:
amazon_directory = '../inputs/'
shopify_directory = '../outputs/'
template_file = '../templates/product_template_shopify.xlsx'

In [3]:
## get_column_mapping
#column_mapping = pd.read_excel(template_file, sheet_name="mapping_old")
column_mapping = pd.read_excel(template_file, sheet_name="mapping")
column_mapping = dict(column_mapping.values)
column_mapping

{'parent_sku': 'Handle',
 'item_name': 'Title',
 'product_description': 'Body (HTML)',
 'color_name': 'Option1 Value',
 'size_name': 'Option2 Value',
 'item_sku': 'Variant SKU',
 'purchasable_offer[marketplace_id=A1F83G8C2ARO7P]#1.our_price#1.schedule#1.value_with_tax': 'Variant Price',
 'main_image_url': 'Variant Image',
 'department_name': 'Google Shopping / Gender',
 'age_range_description': 'Google Shopping / Age Group'}

In [4]:
## get_required_fields
required_fields = pd.read_excel(template_file, sheet_name="field_definitions",
                                skiprows=2)
required_fields = required_fields.drop(columns='Description')
required_fields = required_fields[required_fields['Required']=='required']
required_fields

,Name,Required,Default,Set
0,Handle,required,NaN,NaN
1,Vendor,required,NaN,Shirtshack
2,Published,required,true,true
3,Option1 Name,required,Title,Color
4,Option1 Value,required,Default Title,NaN
6,Variant Grams,required,0,180
7,Variant Inventory Qty,required,0,NaN
8,Variant Inventory Policy,required,deny,NaN
9,Variant Fulfillment Service,required,manual,NaN
10,Variant Price,required,0,NaN


In [5]:
## get_set_fields
set_fields = pd.read_excel(template_file, sheet_name="field_definitions",
                           skiprows=2)
set_fields = set_fields.drop(columns='Description')
set_fields = set_fields[set_fields['Set'].notna()]
set_fields

,Name,Required,Default,Set
1,Vendor,required,NaN,Shirtshack
2,Published,required,true,true
3,Option1 Name,required,Title,Color
6,Variant Grams,required,0,180
14,Variant Weight Unit,required,kg,g
20,Product Category,NaN,NaN,Apparel & Accessories > Clothing
21,Type,NaN,NaN,Shirts
22,Tags,NaN,NaN,t-shirt
23,Option2 Name,NaN,NaN,Size


In [6]:
shopify_only_fields = pd.concat([required_fields.set_index('Name'),
                                 set_fields.set_index('Name')])
for name in column_mapping.values():
    if name in shopify_only_fields.index:
        shopify_only_fields.drop(index=name, inplace=True)
shopify_only_fields = shopify_only_fields.reset_index()
shopify_only_fields.drop_duplicates(inplace=True)
shopify_only_fields

,Name,Required,Default,Set
0,Vendor,required,NaN,Shirtshack
1,Published,required,true,true
2,Option1 Name,required,Title,Color
3,Variant Grams,required,0,180
4,Variant Inventory Qty,required,0,NaN
5,Variant Inventory Policy,required,deny,NaN
6,Variant Fulfillment Service,required,manual,NaN
7,Variant Requires Shipping,required,true,NaN
8,Variant Taxable,required,true,NaN
9,Gift Card,required,false,NaN


In [7]:
shopify_only_fields['Set'] = shopify_only_fields['Set'].fillna(shopify_only_fields['Default'])
shopify_only_fields.drop(columns='Required', inplace=True)
shopify_only_fields.drop(columns='Default', inplace=True)
shopify_only_fields = shopify_only_fields.T
shopify_only_fields.columns = shopify_only_fields.iloc[0]
shopify_only_fields.reset_index(drop=True, inplace=True)
shopify_only_fields.drop(index=0, inplace=True)
shopify_only_fields

Name,Vendor,Published,Option1 Name,Variant Grams,Variant Inventory Qty,Variant Inventory Policy,Variant Fulfillment Service,Variant Requires Shipping,Variant Taxable,Gift Card,Variant Weight Unit,Included / [Primary],Included / International,Status,Product Category,Type,Tags,Option2 Name
1,Shirtshack,true,Color,180,0,deny,manual,true,true,false,g,true,true,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size


In [8]:
shopify_drop_columns = ['Variant Inventory Qty',
                        'Included / [Primary]',
                        'Included / International',
                        ]

for column in shopify_drop_columns:
    if column in shopify_only_fields.columns:
        shopify_only_fields.drop(columns=column, inplace=True)
shopify_only_fields

Name,Vendor,Published,Option1 Name,Variant Grams,Variant Inventory Policy,Variant Fulfillment Service,Variant Requires Shipping,Variant Taxable,Gift Card,Variant Weight Unit,Status,Product Category,Type,Tags,Option2 Name
1,Shirtshack,true,Color,180,deny,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size


In [9]:
## process_files
if not os.path.exists(shopify_directory):
    os.makedirs(shopify_directory)

for filename in os.listdir(amazon_directory):
    if filename.endswith((".csv", ".tsv")):  # Check if the file is a CSV or TSV
        amazon_file_path = os.path.join(amazon_directory, filename)
        shopify_file_path = os.path.join(shopify_directory,
                                         os.path.splitext(filename)[0] + '_shopified_2.csv')

In [10]:
(amazon_file_path, shopify_file_path)

('../inputs/bulk-products-test-adult-t_.tsv',
 '../outputs/bulk-products-test-adult-t__shopified_2.csv')

In [11]:
## Convert Amazon to Shopify

# Determine file extension to choose appropriate read function
file_extension = os.path.splitext(amazon_file_path)[1].lower()
if file_extension == '.csv':
    amazon_df = pd.read_csv(amazon_file_path, header=1)
elif file_extension == '.tsv':
    amazon_df = pd.read_csv(amazon_file_path, sep='\t', header=1)
else:
    raise ValueError("Unsupported file format. Only .csv and .tsv files are supported.")

# Drop the first row which is now the unwanted second row in the original file
#amazon_df = amazon_df.drop(amazon_df.index[0])

# Set unique column names
amazon_df.columns = amazon_df.iloc[0].values
amazon_df.drop(index=0, inplace=True)
amazon_df.reset_index(drop=True, inplace=True)
amazon_df.head()

,feed_product_type,item_sku,brand_name,update_delete,external_product_id,external_product_id_type,item_name,product_description,recommended_browse_nodes,outer_material_type,...,progressive_discount_type,progressive_discount_lower_bound1,progressive_discount_value1,progressive_discount_lower_bound2,progressive_discount_value2,progressive_discount_lower_bound3,progressive_discount_value3,national_stock_number,unspsc_code,pricing_action
0,shirt,Funny-Ghost-T,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,shirt,Funny-Ghost-T-Black-S,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,shirt,Funny-Ghost-T-Black-M,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,shirt,Funny-Ghost-T-Black-L,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,shirt,Funny-Ghost-T-Black-XL,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Drop rows with no 'Parent SKU'
no_psku_rows = amazon_df['parent_sku'].isna()
no_psku_idx = np.where(no_psku_rows)[0]
if len(no_psku_idx) > 0:
    amazon_df = amazon_df[~no_psku_rows]
    no_psku_idx = no_psku_idx + 4
    #logger.warning(
    #                f"The following rows in {amazon_file_path} have no 'Parent SKU' values and were eliminated: {no_psku_idx}"
    #            )
    print(f"The following rows in {amazon_file_path} have no 'parent_sku' values and were eliminated: {no_psku_idx}")
amazon_df.reset_index(drop=True, inplace=True)
amazon_df.head()

The following rows in ../inputs/bulk-products-test-adult-t_.tsv have no 'parent_sku' values and were eliminated: [  4  37  70 103 136]


,feed_product_type,item_sku,brand_name,update_delete,external_product_id,external_product_id_type,item_name,product_description,recommended_browse_nodes,outer_material_type,...,progressive_discount_type,progressive_discount_lower_bound1,progressive_discount_value1,progressive_discount_lower_bound2,progressive_discount_value2,progressive_discount_lower_bound3,progressive_discount_value3,national_stock_number,unspsc_code,pricing_action
0,shirt,Funny-Ghost-T-Black-S,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,shirt,Funny-Ghost-T-Black-M,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,shirt,Funny-Ghost-T-Black-L,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,shirt,Funny-Ghost-T-Black-XL,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,shirt,Funny-Ghost-T-Black-XXL,The Shirt Shack,NaN,NaN,NaN,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,1731138031,Cotton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Add bullet points to product description (to "fluff" it up)
for i in range(amazon_df.shape[0]):
    for j in range(5): # In the current version, 5 bullet points are generated
        amazon_df.loc[i,'product_description'] = (amazon_df.loc[i,'product_description']
                                               + '\n<p>'
                                               + amazon_df.loc[i, 'bullet_point'+str(j+1)]
                                               + '</p>')

In [14]:
# Filter and rename the necessary columns
if not set(column_mapping.keys()).issubset(amazon_df.columns):
    missing_columns = set(column_mapping.keys()) - set(amazon_df.columns)
    # Add logger.warning ???
    raise ValueError(f"Missing columns in {amazon_file_path}: {missing_columns}")
shopify_df = amazon_df[list(column_mapping.keys())].rename(columns=column_mapping)
shopify_df.reset_index(drop=True, inplace=True)
shopify_df.head()

,Handle,Title,Body (HTML),Option1 Value,Option2 Value,Variant SKU,Variant Price,Variant Image,Google Shopping / Gender,Google Shopping / Age Group
0,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,S,Funny-Ghost-T-Black-S,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult
1,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,M,Funny-Ghost-T-Black-M,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult
2,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,L,Funny-Ghost-T-Black-L,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult
3,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XL,Funny-Ghost-T-Black-XL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult
4,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XXL,Funny-Ghost-T-Black-XXL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult


In [15]:
shopify_only_fields = shopify_only_fields.loc[shopify_only_fields.index.repeat(shopify_df.shape[0])]
shopify_only_fields.reset_index(drop=True, inplace=True)
shopify_only_fields.shape

(160, 15)

In [16]:
shopify_df = pd.concat([shopify_df, shopify_only_fields], axis=1)
shopify_df.head()

,Handle,Title,Body (HTML),Option1 Value,Option2 Value,Variant SKU,Variant Price,Variant Image,Google Shopping / Gender,Google Shopping / Age Group,...,Variant Fulfillment Service,Variant Requires Shipping,Variant Taxable,Gift Card,Variant Weight Unit,Status,Product Category,Type,Tags,Option2 Name
0,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,S,Funny-Ghost-T-Black-S,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size
1,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,M,Funny-Ghost-T-Black-M,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size
2,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,L,Funny-Ghost-T-Black-L,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size
3,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XL,Funny-Ghost-T-Black-XL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size
4,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XXL,Funny-Ghost-T-Black-XXL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,manual,true,true,false,g,active,Apparel & Accessories > Clothing,Shirts,t-shirt,Size


In [17]:
# Drop items with no 'Handle' or no 'Title'
no_handle_rows = shopify_df['Handle'].isna()
no_handle_idx = np.where(no_handle_rows)[0]
if len(no_handle_idx) > 0:
    shopify_df = shopify_df[~no_handle_rows]
    no_handle_idx = no_handle_idx + 4
    #logger.warning(
    #                f"The following rows in {amazon_file_path} produced empty 'Handle' values: {no_handle_idx}"
    #            )
    print(f"The following rows in {amazon_file_path} produced empty 'Handle' values: {no_handle_idx}")

no_title_rows = shopify_df['Title'].isna()
no_title_idx = np.where(no_title_rows)[0]
if len(no_title_idx) > 0:
    shopify_df = shopify_df[~no_title_rows]
    no_title_idx = no_title_idx + 4
    #logger.warning(
    #                f"The following rows in {amazon_file_path} produced empty 'Title' values: {no_title_idx}"
    #            )
    print(f"The following rows in {amazon_file_path} produced empty 'Title' values: {no_title_idx}")

In [18]:
# Specify column names to read from the Amazon file for images
image_column_names = [
    'main_image_url','other_image_url1', 'other_image_url2', 'other_image_url3', 
    'other_image_url4', 'other_image_url5', 'other_image_url6', 
    'other_image_url7' , 'other_image_url8'
]

# Initialize the position counter
image_position_counter = 1

# Add 'Image Src', 'Image Position', and 'Image Alt Text' columns to the DataFrame
image_src_col_names = []
for column_name in image_column_names:
    image_src_col_name = f'Image Src {image_position_counter}'
    image_pos_col_name = f'Image Position {image_position_counter}'
    image_alt_text_col_name = f'Image Alt Text {image_position_counter}'
    
    # Check if the image URL column is empty and then if it contains ".db" or
    # "size-guide". If not, add it to the DataFrame
    #if not amazon_df[column_name].str.contains('.db|size-guide').any():
    if amazon_df[column_name].any():
        if not amazon_df[column_name].str.contains('.db|size-guide').any():
            shopify_df[image_src_col_name] = amazon_df[column_name]
            shopify_df[image_pos_col_name] = image_position_counter
            shopify_df[image_alt_text_col_name] = shopify_df['Option1 Value']
            image_src_col_names.append(image_src_col_name)
            image_position_counter += 1

# Replace empty image URLs with main image URL
for column_name in image_src_col_names:
    shopify_df[column_name] = shopify_df[column_name].fillna(shopify_df['Variant Image'])

# Rename the temporary columns to their final names
for i in range(1, image_position_counter):
    shopify_df.rename(columns={
        f'Image Src {i}': 'Image Src',
        f'Image Position {i}': 'Image Position',
        f'Image Alt Text {i}': 'Image Alt Text'
    }, inplace=True)

shopify_df.head()

,Handle,Title,Body (HTML),Option1 Value,Option2 Value,Variant SKU,Variant Price,Variant Image,Google Shopping / Gender,Google Shopping / Age Group,...,Image Alt Text,Image Src,Image Position,Image Alt Text,Image Src,Image Position,Image Alt Text,Image Src,Image Position,Image Alt Text
0,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,S,Funny-Ghost-T-Black-S,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,Black,https://storage.googleapis.com/shirtshack-imag...,3,Black,https://storage.googleapis.com/shirtshack-imag...,4,Black,https://storage.googleapis.com/shirtshack-imag...,5,Black
1,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,M,Funny-Ghost-T-Black-M,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,Black,https://storage.googleapis.com/shirtshack-imag...,3,Black,https://storage.googleapis.com/shirtshack-imag...,4,Black,https://storage.googleapis.com/shirtshack-imag...,5,Black
2,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,L,Funny-Ghost-T-Black-L,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,Black,https://storage.googleapis.com/shirtshack-imag...,3,Black,https://storage.googleapis.com/shirtshack-imag...,4,Black,https://storage.googleapis.com/shirtshack-imag...,5,Black
3,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XL,Funny-Ghost-T-Black-XL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,Black,https://storage.googleapis.com/shirtshack-imag...,3,Black,https://storage.googleapis.com/shirtshack-imag...,4,Black,https://storage.googleapis.com/shirtshack-imag...,5,Black
4,Funny-Ghost-T,"The Shirt Shack Funny Ghost ""I Need to Be Samb...",<p>Lighten the Mood with a Touch of Humor</p>\...,Black,XXL,Funny-Ghost-T-Black-XXL,11.99,https://storage.googleapis.com/shirtshack-imag...,Unisex,Adult,...,Black,https://storage.googleapis.com/shirtshack-imag...,3,Black,https://storage.googleapis.com/shirtshack-imag...,4,Black,https://storage.googleapis.com/shirtshack-imag...,5,Black


In [19]:
# Remove brand name from titles
brand_name = amazon_df['brand_name'].unique()[0] + ' '
shopify_df['Title'] = shopify_df['Title'].str.replace(brand_name, '')

In [20]:
# Populate Tags field

field_for_tags = ['Tags',
                  'Option1 Value',
                  'Google Shopping / Age Group',
                  'Google Shopping / Gender',
                  'Handle',
                ]

for i in range(shopify_df.shape[0]):
    tags = []
    for keyword in amazon_df.loc[i, 'generic_keywords'].split(', '):
        tags.append(keyword.replace(' ', '-'))
    for field in field_for_tags:
        tags.append(shopify_df.loc[i, field].replace(' ', '-'))
    shopify_df.loc[i, 'Tags'] = '"' + '","'.join(tags) + '"'

In [21]:
# Write the shopified dataframe to a new CSV file
shopify_df.to_csv(shopify_file_path, index=False)
print(f"Processed and saved: {shopify_file_path}")

Processed and saved: ../outputs/bulk-products-test-adult-t__shopified_2.csv


### Playground for taking a buffer as in input instead of a csv/tsv file

In [ ]:
import io

amazon_file_path = '../inputs/auto-race-amazon.csv'
buff = io.StringIO()

with open(amazon_file_path) as file:
    for line in file:
        buff.write(line)

buff.seek(0)
xx = pd.read_csv(buff, header=1)

yy = pd.read_csv(amazon_file_path, header=1)

xx.equals(yy)